# Data Preparation: BNP Paribas Cardif Claims

Downloads the BNP Paribas Cardif Claims Management dataset from OpenML, filters labeled rows, splits 80/20, and saves train/score tables to Unity Catalog.

**Compute:** Serverless CPU

In [0]:
%pip install openml --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import os
import sys

# Make the repo-level common/ package importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import openml

In [0]:
# Download BNP Paribas Cardif Claims Management dataset from OpenML
dataset = openml.datasets.get_dataset(46625)
df, y, _, _ = dataset.get_data(dataset_format="dataframe")

# Cast unsigned int types to signed (Spark doesn't support uint via Arrow)
for col in df.select_dtypes(include=[np.uint8, np.uint16, np.uint32, np.uint64]).columns:
    df[col] = df[col].astype(np.int32)
# Cast remaining non-float numeric to float64 for safety
for col in df.select_dtypes(include=["category"]).columns:
    df[col] = df[col].astype(str)

print(f"Dataset: {dataset.name}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

# Keep only rows with a target value, then split (80/20)
df_labeled = df.dropna(subset=["target"])
print(f"Labeled rows: {len(df_labeled):,} (dropped {len(df) - len(df_labeled):,} with null target)")

df_train, df_score = train_test_split(df_labeled, test_size=0.2, random_state=42)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "bnp_paribas_cardif_train"
TABLE_SCORE = "bnp_paribas_cardif_score"

spark.createDataFrame(df_train).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")